# 08 — Training Neural Networks II: Optimizers, Schedules, Regularization

**CS231n Lecture 7.** Notebook 03 used one update rule: `w -= lr * dw`. It
works, and it is slow. This notebook builds the five rules that replaced it,
the schedules that squeeze the last few points out, and the regularization that
decides whether any of it generalizes.

Every update rule shares one signature — the same one PyTorch uses:

```python
next_w, config = rule(w, dw, config)
```

`config` carries both the hyperparameters and the optimizer's own **state**
(velocity, running averages, step count). The rule itself is stateless, which
is what lets you swap one for another in a single line.

| You write | |
|---|---|
| `sgd_momentum`, `nesterov_momentum` | accumulate a direction |
| `adagrad`, `rmsprop` | per-parameter step sizes |
| `adam` | both, plus bias correction |
| `lr_*` schedules | step, exponential, inverse, cosine |
| `dropout_forward/backward` | inverted dropout, train vs test |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (get_cifar10, rel_error, grade, show_solution, use,
                          eval_numerical_gradient_array, plot_series,
                          setup_plots, SERIES)

setup_plots()
np.random.seed(0)

data = get_cifar10(num_train=5000, num_val=500, num_test=500,
                   flatten=True, subtract_mean=True)
pixel_std = data['X_train'].std()
X_train = data['X_train'] / pixel_std
X_val   = data['X_val']   / pixel_std
X_test  = data['X_test']  / pixel_std
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']
print("X_train", X_train.shape)

## 1. Why plain SGD is slow

Two failure modes, both geometric.

**Ill-conditioning.** When the loss surface is much steeper in one direction
than another, SGD takes tiny steps along the shallow direction (where it needs
to travel far) and oscillates across the steep one (where it needs to barely
move). You get a zig-zag that makes almost no progress along the axis that
matters. In high dimensions this is the *normal* case, not an edge case.

**Saddle points.** In high dimensions, points with zero gradient are
overwhelmingly saddles rather than minima: a critical point is a local minimum
only if *all* ~10⁶ eigenvalues are positive, and that is vanishingly unlikely.
Near a saddle the gradient is tiny in most directions and SGD crawls. Local
minima are not the problem people imagine; plateaus and saddles are.

## 2. Momentum

$$ v \leftarrow \mu v - \alpha \nabla L \qquad w \leftarrow w + v $$

Think of it as velocity rather than position: the gradient is a *force*, not a
displacement. Consistent gradient components accumulate; oscillating ones
cancel. That directly attacks the zig-zag, and it carries you through small
gradients near saddles.

`mu = 0.9` is the default; `0.99` for a heavier ball.

### Nesterov

Plain momentum computes the gradient where you are. Nesterov computes it where
momentum is **about to take you** — a look-ahead. Since you know you are going
to move by `mu * v` regardless, you may as well evaluate the gradient there and
start correcting one step earlier. The reformulated version used here avoids
having to evaluate the gradient at a different point:

```
v_prev = v
v      = mu * v - lr * dw
w     += -mu * v_prev + (1 + mu) * v
```

### TODO 1 — momentum and Nesterov

Store the velocity in `config['velocity']` so it persists across calls.

In [ ]:
def sgd_momentum(w, dw, config=None):
    """-> (next_w, config).  Uses config['learning_rate'], config['momentum']."""
    if config is None:
        config = {}
    config.setdefault('learning_rate', 1e-2)
    config.setdefault('momentum', 0.9)
    v = config.get('velocity', np.zeros_like(w))

    next_w = None
    # TODO: update v, then w; write v back into config['velocity']

    return next_w, config


def nesterov_momentum(w, dw, config=None):
    """-> (next_w, config).  Use the reformulated update above."""
    if config is None:
        config = {}
    config.setdefault('learning_rate', 1e-2)
    config.setdefault('momentum', 0.9)
    v = config.get('velocity', np.zeros_like(w))

    next_w = None
    # TODO

    return next_w, config

In [ ]:
grade('nb08', 'sgd_momentum', sgd_momentum, verbose=False)
grade('nb08', 'nesterov_momentum', nesterov_momentum, verbose=False)

## 3. Per-parameter learning rates

**AdaGrad** divides each parameter's step by the square root of the running sum
of its squared gradients. Parameters that have seen big gradients get smaller
steps; rarely-updated parameters keep large ones.

The flaw is structural: that sum only ever **grows**, so the effective learning
rate decays monotonically to zero. On a convex problem that is a feature. On a
deep net it means training stops before it should.

**RMSProp** changes one character — the accumulator becomes a *leaky* moving
average:

```
cache = decay * cache + (1 - decay) * dw**2
```

Now the accumulator can shrink again, so the step size recovers. `decay = 0.99`
is standard.

### TODO 2 — AdaGrad and RMSProp

In [ ]:
def adagrad(w, dw, config=None):
    if config is None:
        config = {}
    config.setdefault('learning_rate', 1e-2)
    config.setdefault('epsilon', 1e-8)
    cache = config.get('cache', np.zeros_like(w))

    next_w = None
    # TODO: accumulate dw**2 into cache, then scale the step by 1/(sqrt(cache)+eps)

    return next_w, config


def rmsprop(w, dw, config=None):
    if config is None:
        config = {}
    config.setdefault('learning_rate', 1e-2)
    config.setdefault('decay_rate', 0.99)
    config.setdefault('epsilon', 1e-8)
    cache = config.get('cache', np.zeros_like(w))

    next_w = None
    # TODO: same, but with a LEAKY accumulator

    return next_w, config

In [ ]:
grade('nb08', 'adagrad', adagrad, verbose=False)
grade('nb08', 'rmsprop', rmsprop, verbose=False)

## 4. Adam — and the bias correction everyone forgets

Adam = momentum (first moment `m`) + RMSProp (second moment `v`) + a correction.

```
t += 1
m = beta1*m + (1-beta1)*dw
v = beta2*v + (1-beta2)*dw**2
m_hat = m / (1 - beta1**t)          <-- bias correction
v_hat = v / (1 - beta2**t)          <-- bias correction
w -= lr * m_hat / (sqrt(v_hat) + eps)
```

**Why the correction exists.** `m` and `v` both start at **zero**, so both are
biased toward zero early on. After one step with a constant gradient,
`m = 0.1*dw` (10x too small) and `v = 0.001*dw²` (1000x too small).

The two biases do **not** cancel, because `v` sits under a square root in the
denominator. The net distortion of the step is

$$ \frac{\text{uncorrected}}{\text{corrected}} = \frac{\sqrt{1-\beta_2^t}}{1-\beta_1^t} $$

which at `t=1` is $\sqrt{0.001}/0.1 \approx 0.32$ — so the **uncorrected step
is about 3x too LARGE**, not too small, and it stays too large for a while.
You will measure this in the next cell.

Dividing by `1 - beta**t` removes exactly that startup bias; both factors decay
to 1 as `t` grows, so the correction quietly disappears once training is going.
It is the most commonly omitted part of a from-scratch Adam.

**Defaults that are almost always right:** `beta1=0.9`, `beta2=0.999`,
`eps=1e-8`, `lr=1e-3`. Adam with these is the correct first thing to try on
almost any new problem.

### TODO 3 — Adam

In [ ]:
def adam(w, dw, config=None):
    if config is None:
        config = {}
    config.setdefault('learning_rate', 1e-3)
    config.setdefault('beta1', 0.9)
    config.setdefault('beta2', 0.999)
    config.setdefault('epsilon', 1e-8)
    m = config.get('m', np.zeros_like(w))
    v = config.get('v', np.zeros_like(w))
    t = config.get('t', 0)

    next_w = None
    # TODO: increment t FIRST, then m, v, the two corrections, then the step.
    # Write m, v and t back into config.

    return next_w, config

In [ ]:
grade('nb08', 'adam', adam, verbose=False)

In [ ]:
_adam = use('nb08', 'adam', adam)

# Constant gradient of 1, lr 0.1. A correctly bias-corrected Adam should move
# almost exactly 0.1 per step, right from step 1.
w = np.zeros(3)
cfg = {'learning_rate': 0.1}
corrected = []
for step in range(1, 8):
    prev = w.copy()
    w, cfg = _adam(w, np.ones(3), cfg)
    corrected.append(abs(w[0] - prev[0]))

# the same rule with the two correction terms removed
b1, b2, lr_, eps_ = 0.9, 0.999, 0.1, 1e-8
m_, v_ = 0.0, 0.0
uncorrected = []
for step in range(1, 8):
    m_ = b1 * m_ + (1 - b1) * 1.0
    v_ = b2 * v_ + (1 - b2) * 1.0
    uncorrected.append(lr_ * m_ / (np.sqrt(v_) + eps_))

print("   t | corrected | uncorrected | ratio")
for t, (c, u) in enumerate(zip(corrected, uncorrected), start=1):
    print("  %2d |  %.5f  |   %.5f   | %.2fx too large" % (t, c, u, u / c))
print("\nThe corrected step is a clean 0.1 throughout, while the uncorrected")
print("one overshoots by 3-6x during the startup phase.")

### See the difference on a badly-conditioned surface

$f(x,y) = 0.05x^2 + y^2$ — twenty times steeper in `y` than in `x`. The classic
zig-zag test.

In [ ]:
mom  = use('nb08', 'sgd_momentum', sgd_momentum)
nest = use('nb08', 'nesterov_momentum', nesterov_momentum)
ada  = use('nb08', 'adagrad', adagrad)
rms  = use('nb08', 'rmsprop', rmsprop)

from solutions.nb08 import sgd as _sgd

def f2(x, y):
    return 0.05 * x ** 2 + y ** 2

def g2(p):
    return np.array([0.1 * p[0], 2.0 * p[1]])

rules = [
    ('SGD',       _sgd, {'learning_rate': 0.1}),
    ('Momentum',  mom,  {'learning_rate': 0.1, 'momentum': 0.9}),
    ('Nesterov',  nest, {'learning_rate': 0.1, 'momentum': 0.9}),
    ('AdaGrad',   ada,  {'learning_rate': 0.5}),
    ('RMSProp',   rms,  {'learning_rate': 0.1, 'decay_rate': 0.99}),
    ('Adam',      _adam, {'learning_rate': 0.3}),
]

paths = {}
for nm, rule, cfg in rules:
    p, c = np.array([-2.0, 1.0]), dict(cfg)
    pts = [p.copy()]
    for _ in range(60):
        p, c = rule(p, g2(p), c)
        pts.append(p.copy())
    paths[nm] = np.array(pts)

gx = np.linspace(-2.5, 1.2, 200)
gy = np.linspace(-1.3, 1.3, 200)
GX, GY = np.meshgrid(gx, gy)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.contour(GX, GY, f2(GX, GY), levels=18, colors='#d9d8d4', linewidths=0.8)
for i, (nm, pts) in enumerate(paths.items()):
    ax.plot(pts[:, 0], pts[:, 1], color=SERIES[i], linewidth=1.6, label=nm)
    ax.plot(pts[-1, 0], pts[-1, 1], marker='o', markersize=5, color=SERIES[i])
ax.plot(0, 0, marker='*', markersize=12, color='#0b0b0b', zorder=5)
ax.set_title("60 steps on f(x,y) = 0.05x² + y²   (star = optimum)")
ax.set_xlabel("x  (shallow direction)"); ax.set_ylabel("y  (steep direction)")
ax.legend(ncol=3, fontsize=8)
ax.grid(False)
plt.show()

for nm, pts in paths.items():
    print("  %-9s after 60 steps: f = %.4e" % (nm, f2(pts[-1, 0], pts[-1, 1])))

Plain SGD is the one still crawling along the `x` axis — it damped the `y`
oscillation quickly but has barely moved in the direction that actually
matters. The momentum methods overshoot and come back; the adaptive methods go
more or less straight in, because they rescale each axis by its own gradient
history.

## 5. On a real network

The synthetic surface is a cartoon. Here is the same comparison on the
2-layer net, same initialization, same data, 8 epochs each.

In [ ]:
from solutions.nb08 import DropoutNet, train_with_rule

configs = [
    ('SGD',      _sgd,  {'learning_rate': 1e-2}),
    ('Momentum', mom,   {'learning_rate': 1e-2, 'momentum': 0.9}),
    ('Nesterov', nest,  {'learning_rate': 1e-2, 'momentum': 0.9}),
    ('RMSProp',  rms,   {'learning_rate': 1e-4, 'decay_rate': 0.99}),
    ('Adam',     _adam, {'learning_rate': 1e-3}),
]

results = {}
t0 = time.time()
for nm, rule, cfg in configs:
    model = DropoutNet(3072, 200, 10, reg=0.0, dropout_keep=1.0, seed=0)
    h = train_with_rule(model, X_train, y_train, X_val, y_val, rule, cfg,
                        num_epochs=8, batch_size=200)
    results[nm] = h
    print("%-9s final loss %.4f   train %.3f   val %.3f"
          % (nm, np.mean(h['loss'][-25:]), h['train_acc'][-1], h['val_acc'][-1]))
print("\n%.1f s" % (time.time() - t0))

In [ ]:
def smooth(v, k=25):
    return np.convolve(np.asarray(v, float), np.ones(k) / k, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
n = len(smooth(results['SGD']['loss']))
plot_series(axes[0], np.arange(n),
            [(nm, smooth(h['loss'])[:n]) for nm, h in results.items()],
            "Training loss", "iteration", "loss", direct_labels=False)
plot_series(axes[1], np.arange(1, 9),
            [(nm, h['val_acc']) for nm, h in results.items()],
            "Validation accuracy", "epoch", "accuracy", direct_labels=False)
plt.show()

> **Read this carefully, because it is the honest result.** The adaptive
> methods drive the *training loss* down far faster — that is what an optimizer
> is for. The *validation* accuracies land within a couple of points of each
> other.
>
> **A better optimizer optimizes better. It does not, by itself, generalize
> better.** Faster convergence to a lower training loss can even mean slightly
> worse generalization. Use Adam because it gets you there in fewer epochs and
> needs less learning-rate tuning — not because it will raise your test score.

## 6. Learning rate schedules

A high learning rate explores; a low one refines. Start high, end low.

### TODO 4 — four schedules

In [ ]:
def lr_step_decay(lr0, epoch, drop=0.5, every=5):
    """Multiply by `drop` every `every` epochs (a step function)."""
    pass  # TODO

def lr_exponential_decay(lr0, epoch, k=0.1):
    """lr0 * exp(-k * epoch)"""
    pass  # TODO

def lr_inverse_decay(lr0, epoch, k=0.1):
    """lr0 / (1 + k * epoch)"""
    pass  # TODO

def lr_cosine(lr0, epoch, total_epochs):
    """Half a cosine from lr0 down to 0 over total_epochs."""
    pass  # TODO

In [ ]:
for nm, fn in [('lr_step_decay', lr_step_decay),
               ('lr_exponential_decay', lr_exponential_decay),
               ('lr_inverse_decay', lr_inverse_decay),
               ('lr_cosine', lr_cosine)]:
    grade('nb08', nm, fn, verbose=False)

In [ ]:
from solutions.nb08 import lr_warmup_cosine

sched = {
    'step':        use('nb08', 'lr_step_decay', lr_step_decay),
    'exponential': use('nb08', 'lr_exponential_decay', lr_exponential_decay),
    'inverse':     use('nb08', 'lr_inverse_decay', lr_inverse_decay),
    'cosine':      use('nb08', 'lr_cosine', lr_cosine),
}

E = 40
eps_ = np.arange(E)
series = [
    ('step',        [sched['step'](0.1, e) for e in eps_]),
    ('exponential', [sched['exponential'](0.1, e) for e in eps_]),
    ('inverse',     [sched['inverse'](0.1, e) for e in eps_]),
    ('cosine',      [sched['cosine'](0.1, e, E) for e in eps_]),
    ('warmup+cosine', [lr_warmup_cosine(0.1, e, E, 5) for e in eps_]),
]
fig, ax = plt.subplots(figsize=(6.6, 3.6))
plot_series(ax, eps_, series, "Learning-rate schedules", "epoch",
            "learning rate", direct_labels=False)
plt.show()

**Cosine is the modern default** for a fixed training budget — smooth, no step
hyperparameters to pick, and it anneals all the way to zero.

**Warmup** (the flat-then-rise start) exists because Adam's second-moment
estimate is unreliable in the first few hundred steps, and because very large
batches produce unstable early gradients. It is essentially mandatory for
Transformers and large-batch training, and harmless otherwise.

> **Tune the learning rate before you tune the schedule.** A good constant rate
> beats a bad rate with a beautiful schedule, every time.

In [ ]:
# Cosine schedule against a constant rate, same budget.
cos = sched['cosine']
comp = {}
for nm, sch in [('constant 1e-3', None),
                ('cosine from 2e-3', lambda e: cos(2e-3, e, 15))]:
    model = DropoutNet(3072, 200, 10, reg=0.0, dropout_keep=1.0, seed=0)
    h = train_with_rule(model, X_train, y_train, X_val, y_val, _adam,
                        {'learning_rate': 1e-3}, num_epochs=15, batch_size=200,
                        lr_schedule=sch)
    comp[nm] = h
    print("%-18s train %.3f  val %.3f" % (nm, h['train_acc'][-1], h['val_acc'][-1]))

## 7. Dropout

At training time, zero each unit independently with probability `1 - p`. At
test time, use the whole network.

**Why it helps (two readings):**
- It prevents **co-adaptation** — no unit can rely on a specific partner being
  present, so features must be individually useful.
- It is an **implicit ensemble** over the exponentially many sub-networks you
  could sample, all sharing weights.

### Inverted dropout — the detail that matters

The naive version scales by `p` at test time to match expectations. The
**inverted** version divides by `p` at *train* time instead:

```
mask = (rand(*x.shape) < p) / p       # note the / p
```

Same expectation, but now **test-time code is completely unchanged** — no
scaling, no branch, no cost. Every framework does it this way, and it is why
`model.eval()` only needs to flip a flag.

### TODO 5 — dropout

Here `p` is the probability of **keeping** a unit. (PyTorch's `nn.Dropout(p)`
uses `p` as the probability of *dropping* — opposite convention, a genuine
source of bugs.)

In [ ]:
def dropout_forward(x, dropout_param):
    """dropout_param: 'p' (keep prob), 'mode' ('train'|'test'), optional 'seed'.

    -> (out, cache).  Return cache = (dropout_param, mask), with mask None in
    test mode. Seed with np.random.seed(dropout_param['seed']) when present.
    """
    p, mode = dropout_param['p'], dropout_param['mode']
    if dropout_param.get('seed') is not None:
        np.random.seed(dropout_param['seed'])

    out, cache = None, None
    # TODO
    return out, cache


def dropout_backward(dout, cache):
    """-> dx.  Gradient flows only through the units that were kept."""
    pass  # TODO

In [ ]:
grade('nb08', 'dropout_forward', dropout_forward, verbose=False)
grade('nb08', 'dropout_backward', dropout_backward, verbose=False)

In [ ]:
df = use('nb08', 'dropout_forward', dropout_forward)
db = use('nb08', 'dropout_backward', dropout_backward)

xd = np.random.RandomState(0).randn(500, 500) + 10
print("  p(keep)   train mean   test mean   fraction zeroed")
for p in [0.25, 0.5, 0.75, 1.0]:
    o_tr, _ = df(xd, {'p': p, 'mode': 'train', 'seed': 1})
    o_te, _ = df(xd, {'p': p, 'mode': 'test', 'seed': 1})
    print("   %.2f      %8.4f    %8.4f       %.3f"
          % (p, o_tr.mean(), o_te.mean(), np.mean(o_tr == 0)))
print("\nTrain and test means agree -- that is exactly what inverting buys.")

xs = np.random.RandomState(4).randn(6, 5)
dp = {'p': 0.6, 'mode': 'train', 'seed': 7}
o, c = df(xs, dict(dp))
d = np.random.RandomState(8).randn(*o.shape)
print("dropout dx rel error %.3e" % rel_error(db(d, c),
      eval_numerical_gradient_array(lambda t: df(t, dict(dp))[0], xs.copy(), d)))

In [ ]:
# Does it actually regularize here? Small training set, large hidden layer --
# the regime where a net can memorise outright.
#
# Run each setting on THREE seeds. A single run cannot distinguish a real
# effect from seed noise, and this is exactly where people fool themselves.
print("500 training examples, 500 hidden units, 25 epochs of Adam, 3 seeds")
print("(this cell takes a few minutes)\n")

drop_res, drop_curves = {}, {}
for keep in [1.0, 0.5, 0.25]:
    vals, gaps = [], []
    for seed in range(3):
        m = DropoutNet(3072, 500, 10, reg=0.0, dropout_keep=keep, seed=seed)
        h = train_with_rule(m, X_train[:500], y_train[:500], X_val, y_val,
                            _adam, {'learning_rate': 5e-4},
                            num_epochs=25, batch_size=50, seed=seed)
        vals.append(h['val_acc'][-1])
        gaps.append(h['train_acc'][-1] - h['val_acc'][-1])
        if seed == 0:
            drop_curves[keep] = h['val_acc']
    drop_res[keep] = (np.mean(vals), np.std(vals), np.mean(gaps), vals)
    print("  keep %.2f -> val %.4f +/- %.4f   gap %.3f   seeds %s"
          % (keep, np.mean(vals), np.std(vals), np.mean(gaps),
             np.round(vals, 3)))

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.6))
keeps = list(drop_res)
means = [drop_res[k][0] for k in keeps]
stds = [drop_res[k][1] for k in keeps]
ax.errorbar(keeps, means, yerr=stds, color=SERIES[0], ecolor='#b9cfeb',
            elinewidth=1.4, capsize=4, marker='o', markersize=5)
for k, m_ in zip(keeps, means):
    ax.annotate("%.3f" % m_, xy=(k, m_), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=8,
                color='#52514e')
ax.set_title("Validation accuracy vs dropout strength (mean ± sd over 3 seeds)")
ax.set_xlabel("keep probability")
ax.set_ylabel("val accuracy")
ax.set_axisbelow(True)
plt.show()

### The honest reading — and it is a negative result

**Dropout does not help here.** The three settings land within about one
standard deviation of each other, and the ordering is not even monotone. The
error bars overlap completely. Whatever a single run appeared to show, three
runs say the effect is indistinguishable from seed noise.

> **Take the methodology lesson, not just the dropout lesson.** A single
> training run comparing two regularizers tells you very little — run-to-run
> variation here is ±1–2 accuracy points, which is the same size as most
> effects people report from a single run. **Always check the seed spread
> before believing a regularization result.** This is the single most common
> way people fool themselves in applied deep learning.

Why dropout underperforms in this setup, concretely:

- Only 500 examples and a 2-layer net. It memorises the training set under
  every setting, so there is little for dropout to prevent.
- Dropout's famous wins were on the enormous fully-connected classifier heads
  of AlexNet and VGG — 4096-unit layers holding the bulk of the parameters.
  There is no such layer here.

Modern convnets are mostly convolutional, use BatchNorm (which regularizes on
its own), and consequently use much less dropout or none at all. Know it,
recognise it in papers, and reach for **data augmentation** and BatchNorm
first. You will measure augmentation's effect on a real convnet in notebook 09.

### The regularization menu, ranked by what usually helps most

| | What it does |
|---|---|
| **More data** | always first, if you can get it |
| **Data augmentation** | free extra data; flips/crops/colour jitter |
| **Early stopping** | free; stop when val accuracy peaks |
| **L2 / weight decay** | prefers diffuse weights |
| **BatchNorm** | normalizes, and the batch noise regularizes |
| **Dropout** | for large fully-connected layers |

All of these inject a bias or some noise at training time and remove it at
test time. That pattern *is* what regularization is.

## 8. Recap

1. Name the two geometric reasons plain SGD is slow.
2. Why are saddle points, not local minima, the real obstacle in high dimensions?
3. Momentum: what is the physical reading of `v`, and why does it kill zig-zag?
4. What does Nesterov change, in one sentence?
5. Why does AdaGrad stall, and what one change makes it RMSProp?
6. Write Adam's bias correction and explain what would go wrong without it.
7. An optimizer that reaches a lower training loss faster — does it generalize
   better? What is the actual reason to prefer Adam?
8. Why is cosine the modern default schedule, and what is warmup for?
9. What does "inverted" mean in inverted dropout, and what does it buy?
10. In this notebook `p` is the keep probability. What is it in
    `torch.nn.Dropout`?
11. Why does modern practice use much less dropout than AlexNet did?

In [ ]:
# show_solution('nb08', 'adam')
# show_solution('nb08', 'dropout_forward')